# Práctica 2: Tautomería y protonación como microespacio químico

> **Bloque 1 — Generación de estructuras y espacio conformacional** · Manual de QC · UNAM

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eduardo-Gabriel-Guzman-Lopez/computational-chemistry-book/blob/main/notebooks/p02.ipynb)

In [ ]:
# ── Configuración ─────────────────────────────────────────────
# Google Colab: descomenta la siguiente línea
# !pip install rdkit-pypi xtb-python umap-learn -q

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams["figure.dpi"] = 120
print("Entorno listo ✓  —  Manual QC UNAM")


Tautomería y protonación como microespacio químico

### 📋 Información de la práctica

| Parámetro | Valor |
|:--|:--|
| **Bloque temático** | 1: Generación de estructuras y espacio conformacional |
| **Número de práctica** |  |
| **Nivel de dificultad** | {Básico |
| **Tiempo estimado** | 1.5 h (semilla) + 2 h (bosque y análisis) |
| **Modalidad** | Individual |
| **Pipeline central** | SMILES $\to$ enumeración de tautómeros/protómeros $\to$
 geometría 3D $\to$ optimización GFN2-xTB $\to$
 $\Delta G$ relativo $\to$ población de Boltzmann $\to$
 microespacio químico |
| **Hardware mínimo** | 2 GB RAM, procesador de doble núcleo
 $\geq$ 1.5 GHz |
| **Modo sin conexión** | Parcial — internet solo para instalación
 inicial |
| **Acceso en nube** | {https://colab.research.google.com/drive/p02_tautomeria |

## Introducción química

Cuando un químico escribe el SMILES de una molécula, implícitamente
elige una forma tautomérica y un estado de protonación. Sin embargo, en
condiciones reales —en solución, en el sitio activo de una enzima, en
la superficie de un catalizador— una misma molécula puede existir
simultáneamente en múltiples formas que se interconvierten rápidamente, ocurre una especiación química.
Esta especiación no es un artefacto computacional: es química real con
consecuencias directas sobre la reactividad, la solubilidad, el espectro
UV-Vis y la afinidad de unión a proteínas cite'abrahamsson2024'.

Los **tautómeros** son isómeros constitucionales que se interconvierten
por migración de un protón acompañada del desplazamiento de un enlace
doble. El par ceto-enol del acetilacetona es un ejemplo conocido
en el ámbito farmacéutico, la tautomería es ubicua en bases nucleicas,
heterociclos nitrogenados y compuestos con grupos amidina o guanidina.
Los **protómeros** son las distintas formas que adopta una misma
molécula al protonarse en diferentes sitios básicos. Para un compuesto
con dos centros básicos de pKa cercanos, ambos protómeros pueden estar
presentes en equilibrio a pH fisiológico.

La importancia computacional de este problema es doble. Primero, si el
cálculo de estructura electrónica se ejecuta sobre la forma tautómera
incorrecta, todos los resultados subsecuentes —energías, propiedades,
descriptores— son los de una especie que puede ser minoritaria o
incluso inexistente en las condiciones de interés. Segundo, la
enumeración sistemática de tautómeros y protómeros constituye el primer
ejemplo de *microespacio químico*: un conjunto pequeño pero bien
definido de variantes estructurales derivadas de una misma molécula,
que ya puede analizarse estadísticamente. Abrahamsson (2024) demostró
que la distribución de poblaciones de tautómeros calculadas con GFN2-xTB
reproduce cuantitativamente las respuestas relativas observadas en
espectrometría de masas con ionización por electrospray, validando
el nivel de teoría para este tipo de problemas cite'abrahamsson2024'.

En términos del modelo Semilla–Bosque: la **semilla** es la
enumeración y optimización de todos los tautómeros y protómeros de una
molécula farmacológicamente relevante elegida por el estudiante. El
**bosque** es un dataset de 40 moléculas con tautomería conocida
—incluyendo bases nucleicas, antihistamínicos, inhibidores de quinasa
y colorantes— para las cuales el mismo pipeline ya fue ejecutado.
El análisis del bosque permite identificar qué familias estructurales
son más propensas a la tautomería, cuántas formas coexisten típicamente
a temperatura ambiente y cómo se distribuyen las poblaciones de Boltzmann.


 SMILES semilla,
 Enumeración tautómeros (RDKit),
 Geometría 3D (ETKDG),
 Opt GFN2-xTB,
 $\Delta G$ relativo,
 Población de Boltzmann,
 Microespacio químico

## Marco teórico

### Conceptos clave

**1. Tautomería y el equilibrio tautómero.**
Dos estructuras son tautómeros si pueden interconvertirse mediante la
migración de un protón (o de H$^-$ en algunos casos) con desplazamiento
concomitante de uno o más enlaces dobles. El equilibrio entre las formas
$T_1$ y $T_2$ está gobernado por la diferencia de energía libre estándar:

$$
    K_T = \frac{[T_2]}{[T_1]}
    = \exp\!\left(-\frac{\Delta G^\circ_{T_1 \to T_2}}{RT}\right)
$$

A temperatura ambiente ($T = 298$~K), una diferencia $\Delta G^\circ$ de
tan solo 5.7~kJ/mol ($\approx 1.4$ kcal/mol) implica que la forma
minoritaria representa $\sim 10$\
17~kJ/mol la reduce al 0.1\

**2. Distribución de Boltzmann sobre un microespacio.**
Cuando hay $n$ tautómeros con energías libres $G_i$ (calculadas a la
misma temperatura $T$), la fracción molar de cada uno es:

$$
    p_i = \frac{e^{-G_i/RT}}{\displaystyle\sum_{j=1}^{n} e^{-G_j/RT}}
$$

Esta expresión convierte un conjunto de energías absolutas en una
distribución de probabilidad. Es la conexión central entre los números
del cálculo y la química observable: $p_i$ es la fracción de moléculas
que se encuentran en la forma $i$ en el equilibrio termodinámico.

**3. El algoritmo TautomerEnumerator de RDKit.**
RDKit implementa la enumeración de tautómeros siguiendo las reglas de
transformación de Sitzmann (SMARTS-based). El procedimiento es:
(i) identificar todos los patrones reactivos presentes en la molécula,
(ii) aplicar las transformaciones válidas de forma combinatoria,
(iii) eliminar duplicados canónicos. El número de tautómeros crece
exponencialmente con el número de grupos tautomerizables, por lo que
RDKit limita por defecto la enumeración a 1000 formas.

**4. Protonación y pKa implícito.**
Un protómero se genera añadiendo o quitando un protón en un átomo
con carga formal distinta de cero o con par de electrones no enlazante.
A diferencia de los tautómeros, los protómeros tienen cargas netas
distintas y no son interconvertibles sin intercambio con el entorno.
Para comparar sus energías hay que fijar el pH de referencia usando
el ciclo termodinámico de Born–Haber (Práctica~30), o bien trabajar
en vacío y asumir que la diferencia energética es una aproximación al
$\Delta$pKa.

**5. GFN2-xTB para termoquímica conformacional y tautómera.**
Las energías libres de Gibbs a temperatura $T$ se obtienen añadiendo
correcciones vibracionales, rotacionales y traslacionales a la energía
electrónica. GFN2-xTB calcula estas correcciones en la aproximación
de gas ideal/oscilador armónico:

$$
    G = E_\mathrm{elec} + E_\mathrm{ZPE}
      + H_\mathrm{vib}(T) - TS_\mathrm{vib}(T)
      + H_\mathrm{rot} + H_\mathrm{trans} - TS_\mathrm{rot}
      - TS_\mathrm{trans}
$$

La bandera '–ohess' de xTB calcula frecuencias en la geometría
optimizada y devuelve $G(T)$ directamente. Para la comparación entre
tautómeros, las contribuciones traslacionales y rotacionales se
cancelan en gran medida, y la diferencia $\Delta G$ queda dominada
por $\Delta E_\mathrm{elec} + \Delta \mathrm{ZPE}$.

### Preguntas previas

1. **(Conceptual)** La guanina tiene cuatro tautómeros  relevantes biológicamente: la forma ceto-amino (dominante en  ADN), la enol-amino, la ceto-imino y la enol-imino. Sin  calcular nada, ¿cuál esperas que sea el más estable y por qué?  ¿Qué consecuencia tendría para el apareamiento de bases si  el tautómero minoritario estuviera presente en el 0.01\  de las moléculas?
2. **(Predictivo)** Para la histamina a pH 7.4, hay dos  sitios de protonación posibles: el nitrógeno del anillo  imidazólico ($\delta$) y el nitrógeno de la cadena etilamina.  ¿Cuál crees que estará más protonado en condiciones fisiológicas?  ¿Cómo afecta eso a su interacción con el receptor H$_1$?
3. **(Crítico)** El pipeline de esta práctica calcula  energías en vacío (fase gas) y aplica la distribución de  Boltzmann. ¿En qué circunstancias crees que este protocolo  daría predicciones incorrectas sobre qué tautómero predomina  en solución acuosa? ¿Qué habría que añadir al pipeline para  corregirlo?

### Recursos de consulta rápida

- **Documentación RDKit — MolStandardize:**  [https://www.rdkit.org/docs/source/rdkit.Chem.MolStandardize.rdMolStandardize.html](https://www.rdkit.org/docs/source/rdkit.Chem.MolStandardize.rdMolStandardize.html)  — contiene la clase 'TautomerEnumerator' y sus opciones.
- **Abrahamsson, D. RSC Adv. 2024:**  [DOI:10.1039/d4ra06695b](https://doi.org/10.1039/d4ra06695b) — referencia ancla de esta práctica.  Describe exactamente el pipeline GFN2-xTB + Boltzmann para  predecir respuestas de ESI-MS en función del estado de protonación.
- **Capítulo 4 de Lewars (2024):** termodinámica estadística  para la distribución de Boltzmann en el contexto de cálculos  computacionales. Disponible en la biblioteca del curso.
- **Tautomer dataset (ChEMBL):**  [https://www.ebi.ac.uk/chembl/](https://www.ebi.ac.uk/chembl/) — la base de datos  ChEMBL almacena los tautómeros canónicos de $\sim 2.4$ millones  de compuestos. Útil para contextualizar el tamaño del bosque  de esta práctica.

## Objetivos de aprendizaje

### Conceptuales

- Distinguir entre tautómeros y protómeros, e identificar los  patrones estructurales que predisponen a la tautomería  (grupos amidina, guanidina, ceto-enol, imino-amino).
- Derivar la distribución de Boltzmann a partir de la definición  de energía libre de Gibbs y aplicarla para calcular fracciones  molares de equilibrio.
- Explicar por qué la forma tautómera incorrecta puede invalidar  todos los cálculos de propiedades subsiguientes.

### Procedimentales

- Enumerar tautómeros y protómeros de una molécula usando  'TautomerEnumerator' de RDKit.
- Ejecutar cálculos de optimización y frecuencias con GFN2-xTB  para obtener $G(298\,\mathrm{K})$ de cada forma.
- Calcular y visualizar poblaciones de Boltzmann sobre un  microespacio de tautómeros.

### Investigación y datos

- Construir un dataset de 40 moléculas con las variables:  nombre, SMILES canónico, número de tautómeros enumerados,  SMILES del tautómero más estable, $\Delta G$ de la forma  dominante respecto a la media, y población $p_1$ del más estable.
- Identificar qué familias estructurales exhiben la mayor  dispersión de poblaciones (mayor incertidumbre tautómera).
- Correlacionar el número de tautómeros enumerados con la  complejidad estructural de la molécula.

## Recursos y prerrequisitos

### Conocimientos previos

- **(Necesario)** Práctica~1 completa: instalación de  RDKit, xtb y el entorno conda; protocolo de incrustación 3D  y optimización GFN2-xTB.
- **(Necesario)** Concepto de isomería constitucional y  equilibrio químico.
- **(Necesario)** Termodinámica básica: energía libre de  Gibbs, constante de equilibrio, relación $\Delta G = -RT\ln K$.
- **(Recomendable)** Química de compuestos heterocíclicos  nitrogenados (piridina, imidazol, pirimidina).

### Software

- **Python** $\geq$ 3.10 con 'rdkit' $\geq$ 2023.03,  'numpy', 'pandas', 'matplotlib',  'seaborn'.  Entorno compartido con la Práctica~1: verbatim conda activate qc-manual verbatim
- **xtb** $\geq$ 6.6 — open-source (LGPL-3).  El mismo que se usó en la Práctica~1.
- **Avogadro2** o **py3Dmol** — visualización 3D,  gratuitos y multiplataforma.

In [ ]:
%%bash

conda activate qc-manual

**xtb** $\geq$ 6.6 — open-source (LGPL-3).
 El mismo que se usó en la Práctica~1.

 **Avogadro2** o **py3Dmol** — visualización 3D,
 gratuitos y multiplataforma.
itemize

```{admonition} 📝 Nota metodológica
:class: note

**Esta práctica no requiere Gaussian, ORCA ni ningún software
comercial.** La enumeración de tautómeros usa únicamente RDKit
(open-source) y la termoquímica usa GFN2-xTB (open-source).
El entorno 'qc-manual' instalado en la Práctica~1 ya incluye
todo el software necesario.
```

```{admonition} ⚠️ Advertencia
:class: warning

**¿Sin instalación local?**
Notebook de Google Colab con todas las dependencias:
[https://colab.research.google.com/drive/p02_tautomeria](https://colab.research.google.com/drive/p02_tautomeria)
```

### Archivos proporcionados

- 'p02\_semilla.py' — script base para la semilla.
- 'p02\_bosque\_smiles.csv' — 40 SMILES con nombre,  familia y referencia bibliográfica de la tautomería conocida.
- 'p02\_bosque\_resultados.csv' — dataset pre-calculado.
- 'p02\_analisis.py' — script base de análisis.
- 'p02\_colab.ipynb' — notebook para Google Colab.

### Recursos computacionales y accesibilidad

### 📋 Información de la práctica

| Parámetro | Valor |
|:--|:--|
| **Semilla — CPU** | 5–15 min en 1 núcleo (depende del número
 de tautómeros) |
| **Bosque — CPU** | $\sim$ 2 h en 1 núcleo (pre-calculado) |
| **RAM mínima absoluta** | 2 GB |
| **Espacio en disco** | $< 200$ MB |
| **Modo sin conexión** | Sí, una vez instalado el entorno conda |
| **Acceso en nube** | {https://colab.research.google.com/drive/p02_tautomeria |

```{admonition} 📝 Nota metodológica
:class: note

**Accesibilidad:** Los scripts generan tablas CSV con nombres de
columna en español y figuras con etiquetas textuales en todos los ejes.
Las estructuras 2D de tautómeros se exportan como imágenes PNG con texto
alternativo descriptivo. Ver 'docs/accesibilidad.md'.
```

## Experimento semilla

### Sistema modelo

La semilla de esta práctica es la **2-piridona** / **2-hidroxipiridina**
('O=c1ccccn1H' / 'Oc1ccccn1', CAS 142-08-5 / 142-08-5),
el par tautómero clásico de la química heterocíclica. Este sistema
fue elegido porque: (i) tiene sólo dos tautómeros relevantes, lo que
permite un análisis manual completo antes de escalar; (ii) la diferencia
energética entre ambas formas ha sido medida experimentalmente y calculada
con múltiples niveles de teoría, ofreciendo una referencia de validación
sólida cite'beak1987'; (iii) el problema es lo suficientemente rico
para ilustrar todos los conceptos del Marco Teórico —equilibrio,
distribución de Boltzmann, efecto del solvente— sin complejidad
combinatoria; y (iv) el sistema es la unidad estructural de la uridina
y la timidina, conectando la práctica con bioquímica relevante.

```{admonition} 📝 Nota metodológica
:class: note

**Alternativa para el estudiante:** Si ya tienes experiencia con
la 2-piridona, puedes usar en cambio la **hipoxantina**
('O=c1[nH]cnc2[nH]ccc12'), base nucleica con cuatro tautómeros
relevantes y mayor riqueza conformacional.
```

### Pregunta química concreta

### Nivel de teoría

### 📋 Información de la práctica

| Parámetro | Valor |
|:--|:--|
| **Enumeración de tautómeros** | RDKit {TautomerEnumerator |
| **Incrustación 3D** | ETKDG v3 ({randomSeed = 42 |
| **Pre-optimización** | MMFF94 (o UFF como fallback) |
| **Optimización + frecuencias** | GFN2-xTB, {–opt tight –ohess |
| **Temperatura** | 298.15 K (estándar) |
| **Solvente** | Vacío (fase gas); Práctica~30 introduce PCM/SMD |
| **Justificación** | Abrahamsson (2024) validó GFN2-xTB para
 distribuciones de poblaciones tautómeras frente a datos de ESI-MS
 {abrahamsson2024 |

### Hipótesis inicial

Basándome en las preguntas previas y en la aromaticidad del anillo,
espero que la forma **2-piridona** (lactama, ceto) sea más estable
que la **2-hidroxipiridina** (enol) en fase gas, porque el anillo
lactámico preserva la conjugación $\pi$ con el par del nitrógeno de
forma más efectiva que el sistema enólico. Sin embargo, en solución
acuosa la forma enólica puede estabilizarse por puentes de hidrógeno
con el disolvente, lo que invertiría el orden. Esta hipótesis es
verificable comparando $\Delta G_\mathrm{gas}$ de ambas formas con los
datos de Beak (1987) cite'beak1987': $ G_gas 
-6.3$~kJ/mol a favor de la 2-piridona.

## Protocolo computacional

### 1. Enumeración de tautómeros con RDKit

In [ ]:
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import AllChem, Draw

# Sistema semilla: usar la forma ceto como punto de partida
# (RDKit normalizará y enumerará a partir de cualquier tautómero)
smiles_entrada = 'O=c1ccccn1H'   # 2-piridona
nombre         = '2-piridona'

mol = Chem.MolFromSmiles(smiles_entrada)
if mol is None:
    raise ValueError('SMILES no válido.')

# Configurar el enumerador
enumerador = rdMolStandardize.TautomerEnumerator()
enumerador.SetMaxTautomers(100)     # límite superior
enumerador.SetRemoveSp3Stereo(True)

# Enumerar
tautomeros = enumerador.Enumerate(mol)
print(f'Tautómeros encontrados: {len(tautomeros)}')

# Mostrar SMILES de cada uno
for i, tau in enumerate(tautomeros):
    smi = Chem.MolToSmiles(tau)
    print(f'  T{i+1}: {smi}')

# Guardar imagen de la rejilla de tautómeros
img = Draw.MolsToGridImage(
    list(tautomeros),
    molsPerRow=4,
    subImgSize=(300, 250),
    legends=[f'T{i+1}' for i in range(len(tautomeros))]
)
img.save(f'{nombre}_tautomeros_2D.png')

### 2. Generación de geometrías 3D para cada tautómero

In [ ]:
import os
from rdkit.Chem import rdmolfiles

os.makedirs(f'{nombre}_tautomeros', exist_ok=True)

for i, tau in enumerate(tautomeros):
    etiqueta = f'{nombre}_T{i+1:02d}'

    # Añadir H y embeber
    tau_h  = Chem.AddHs(tau)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    ok = AllChem.EmbedMolecule(tau_h, params)

    if ok == -1:
        print(f'  {etiqueta}: fallo en incrustación, saltando.')
        continue

    # Pre-optimización MMFF94
    ff = AllChem.MMFFGetMoleculeForceField(
             tau_h, AllChem.MMFFGetMoleculeProperties(tau_h))
    if ff is None:
        ff = AllChem.UFFGetMoleculeForceField(tau_h)
    ff.Minimize(maxIts=2000)

    # Exportar XYZ
    xyz_path = f'{nombre}_tautomeros/{etiqueta}_FF.xyz'
    rdmolfiles.MolToXYZFile(tau_h, xyz_path)
    print(f'  {etiqueta}: geometría FF guardada.')

### 3. Optimización + frecuencias con GFN2-xTB

La bandera '–ohess' calcula frecuencias en la geometría
optimizada en un solo paso y devuelve la energía libre de Gibbs $G(T)$
directamente en el log.

In [ ]:
%%bash

# Desde la terminal — ejecutar para cada tautómero:
cd 2-piridona_tautomeros

for f in *_FF.xyz; do
    base="${f
    xtb "$f" --opt tight --ohess --chrg 0 --uhf 0 --gfn 2 \
        --T 298.15 > "${base}_xtb.out" 2>&1
    # Renombrar geometría final
    [ -f xtbopt.xyz ] && mv xtbopt.xyz "${base}_opt.xyz"
    echo "Completado: $base"
done

O bien desde Python (para integración en el pipeline):

In [ ]:
import subprocess, os

directorio = f'{nombre}_tautomeros'
for archivo in sorted(os.listdir(directorio)):
    if not archivo.endswith('_FF.xyz'):
        continue
    base    = archivo.replace('_FF.xyz', '')
    xyz_in  = os.path.join(directorio, archivo)
    log_out = os.path.join(directorio, f'{base}_xtb.out')

    result = subprocess.run(
        ['xtb', archivo, '--opt', 'tight', '--ohess',
         '--chrg', '0', '--uhf', '0', '--gfn', '2',
         '--T', '298.15'],
        capture_output=True, text=True,
        cwd=directorio
    )
    with open(log_out, 'w') as f:
        f.write(result.stdout)

    # Renombrar geometría final
    opt_src = os.path.join(directorio, 'xtbopt.xyz')
    opt_dst = os.path.join(directorio, f'{base}_opt.xyz')
    if os.path.exists(opt_src):
        os.rename(opt_src, opt_dst)

print('Optimizaciones completadas.')

### 4. Extracción de energías libres de Gibbs

In [ ]:
import re, pandas as pd

directorio = f'{nombre}_tautomeros'
registros  = []

for archivo in sorted(os.listdir(directorio)):
    if not archivo.endswith('_xtb.out'):
        continue
    etiqueta = archivo.replace('_xtb.out', '')

    with open(os.path.join(directorio, archivo)) as f:
        texto = f.read()

    # Extraer energía electrónica total
    m_etot = re.search(
        r'TOTAL ENERGY\s+([-\d.]+)\s+Eh', texto)
    # Extraer energía libre de Gibbs (de --ohess)
    m_gibbs = re.search(
        r'TOTAL FREE ENERGY\s+([-\d.]+)\s+Eh', texto)
    # Verificar convergencia
    convergio = 'GEOMETRY OPTIMIZATION CONVERGED' in texto

    registros.append({
        'etiqueta'       : etiqueta,
        'convergio'      : convergio,
        'E_total_Eh'     : float(m_etot.group(1))   if m_etot   else None,
        'G_298_Eh'       : float(m_gibbs.group(1))  if m_gibbs  else None,
    })

df = pd.DataFrame(registros)

# Calcular dG relativo respecto al tautómero más estable
# (convertir de Hartree a kJ/mol: 1 Eh = 2625.5 kJ/mol)
G_min = df['G_298_Eh'].min()
df['dG_kJ_mol'] = (df['G_298_Eh'] - G_min) * 2625.5

print(df[['etiqueta','convergio','G_298_Eh',
          'dG_kJ_mol']].to_string())

### 5. Cálculo de poblaciones de Boltzmann

In [ ]:
import numpy as np

R = 8.314e-3    # kJ / (mol·K)
T = 298.15      # K

# Calcular poblaciones
df['peso_Boltzmann'] = np.exp(-df['dG_kJ_mol'] / (R * T))
suma = df['peso_Boltzmann'].sum()
df['poblacion']      = df['peso_Boltzmann'] / suma
df['porcentaje']     = df['poblacion'] * 100

# Ordenar por estabilidad
df = df.sort_values('dG_kJ_mol').reset_index(drop=True)

print("\n=== Distribución de Boltzmann ===")
print(df[['etiqueta','dG_kJ_mol',
          'poblacion','porcentaje']].round(4).to_string())
print(f"\nTautómero dominante: {df.loc[0,'etiqueta']}")
print(f"Población:           {df.loc[0,'porcentaje']:.2f}

# Guardar
df.to_csv(f'{nombre}_tautomeros_resultados.csv', index=False)

### 6. Archivos generados

- '2-piridona\_tautomeros\_2D.png' — rejilla de tautómeros.
- '2-piridona\_tautomeros/T01\_opt.xyz',  'T02\_opt.xyz', … — geometrías optimizadas.
- '2-piridona\_tautomeros/T01\_xtb.out',  'T02\_xtb.out', … — logs de xTB.
- '2-piridona\_tautomeros\_resultados.csv' — tabla con  $G$, $\Delta G$ y poblaciones de todos los tautómeros.

## Control de calidad y validación

### Criterios de convergencia

### 📋 Información de la práctica

| Parámetro | Valor |
|:--|:--|
| **Convergencia geométrica** | {GEOMETRY OPTIMIZATION CONVERGED |
| **Frecuencias imaginarias** | 0 para todos los tautómeros
 (mínimos genuinos) |

```{admonition} ⚠️ Advertencia
:class: warning

**Frecuencia imaginaria en un tautómero:** Si un tautómero
tiene una frecuencia imaginaria tras '–ohess', su geometría
es un punto de silla, no un mínimo. Esto ocurre cuando la
incrustación inicial coloca al hidrógeno móvil en una posición que
no es el mínimo real de esa forma. Solución: perturbar manualmente
la coordenada del H y reoptimizar, o usar
'–opt verytight' para una convergencia más estricta.
```

### Diagnósticos de consistencia química

Para cada tautómero optimizado, verificar:

- **Identidad del tautómero:** comprobar que el SMILES  de la geometría optimizada coincide con el SMILES de entrada.  Una tautomerización espontánea durante la optimización  indicaría que la forma de entrada no corresponde a un mínimo  real en la PES de GFN2-xTB.  verbatim # Verificar identidad: cargar el XYZ optimizado con RDKit # y comparar SMILES canónico from rdkit.Chem import rdmolfiles, Chem  mol_opt = rdmolfiles.MolFromXYZFile('T01_opt.xyz') if mol_opt:  smi_opt = Chem.MolToSmiles(  Chem.RemoveHs(mol_opt))  print(f'SMILES calculado: smi_opt')  # Comparar con el SMILES original del tautómero T1 verbatim
- **Coherencia de las distancias C=O y C–OH:**  en la forma ceto (2-piridona) esperar $d$(C=O) $\approx 1.22$~;  en la forma enol (2-hidroxipiridina) esperar  $d$(C–OH) $\approx 1.34$~ y $d$(O–H) $\approx 0.97$~.
- **Número de frecuencias imaginarias = 0** para todos  los tautómeros aceptados.

In [ ]:
# Verificar identidad: cargar el XYZ optimizado con RDKit
# y comparar SMILES canónico
from rdkit.Chem import rdmolfiles, Chem

mol_opt = rdmolfiles.MolFromXYZFile('T01_opt.xyz')
if mol_opt:
    smi_opt = Chem.MolToSmiles(
                  Chem.RemoveHs(mol_opt))
    print(f'SMILES calculado: {smi_opt}')
    # Comparar con el SMILES original del tautómero T1

**Coherencia de las distancias C=O y C–OH:**
 en la forma ceto (2-piridona) esperar $d$(C=O) $\approx 1.22$~;
 en la forma enol (2-hidroxipiridina) esperar
 $d$(C–OH) $\approx 1.34$~ y $d$(O–H) $\approx 0.97$~.

 **Número de frecuencias imaginarias = 0** para todos
 los tautómeros aceptados.
itemize

### Validación frente a referencia experimental

Beak (1987) reportó $K_T = [2-piridona]/[2-hidroxipiridina]
= 330 30$ en ciclohexano (apolar) a 295~K, equivalente a
$\Delta G^\circ = -14.2$~kJ/mol cite'beak1987'. En fase gas, el
experimento de microondas de Beak y Fry (1973) da
$\Delta G^\circ_\mathrm{gas} = -6.3$~kJ/mol a favor de la 2-piridona.

| $p$(2-piridona) en fase gas (\ |
|:--|

## Expansión del espacio químico (el bosque)

### Estrategia de expansión

El bosque contiene 40 moléculas seleccionadas para representar la
diversidad de la tautomería en contextos reales:

- **Bases nucleicas** (8): adenina, guanina, citosina,  uracilo, timina e hipoxantina en sus formas canónicas y  tautómeras relevantes biológicamente.
- **Fármacos con tautomería conocida** (12): omeprazol,  pirimetamina, metotrexato, imatinib (fragmento), entre otros.  Referencia: Kaur \& Choudhury (2021) cite'kaur2021'.
- **Heterociclos simples con equilibrio documentado** (10):  2-piridona, 4-piridona, 2-aminopiridina, pirazol, imidazol,  triazoles, tetrazoles.
- **Compuestos con tautomería ceto-enol no obvia** (10):  $\beta$-cetoésteres, malonaldehído, acetilacetona,  dicetonas cíclicas.

Las variables del bosque son:

| 'smiles\_entrada' | str | — | SMILES de partida |
|:--|:--|:--|:--|
| 'familia' | str | — | categoría estructural |
| 'n\_tautomeros' | int | — | RDKit TautomerEnumerator |
| 'smiles\_T1' | str | — | SMILES del más estable |
| 'dG\_T1\_kJ' | float | kJ/mol | GFN2-xTB; $= 0$ por definición |
| 'dG\_T2\_kJ' | float | kJ/mol | GFN2-xTB ($\Delta G$ vs T1) |
| 'pop\_T1\_pct' | float | \
'pop\_T2\_pct' | float | \
'dG\_ref\_kJ' | float | kJ/mol | valor experimental (si existe) |
| 'referencia' | str | — | clave bibliográfica |

### Script de automatización

In [ ]:
%%bash

#!/usr/bin/env python3
# p02_expansion.py
import os, subprocess, re
import numpy as np, pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, rdmolfiles
from rdkit.Chem.MolStandardize import rdMolStandardize

R = 8.314e-3  # kJ/(mol·K)
T = 298.15    # K

df_in = pd.read_csv('p02_bosque_smiles.csv')
enumerador = rdMolStandardize.TautomerEnumerator()
enumerador.SetMaxTautomers(100)

resultados = []

for _, fila in df_in.iterrows():
    nombre_mol = fila['id']
    mol = Chem.MolFromSmiles(fila['smiles_entrada'])
    if mol is None:
        continue

    tautomeros = enumerador.Enumerate(mol)
    reg = {'id': nombre_mol,
           'smiles_entrada': fila['smiles_entrada'],
           'familia': fila['familia'],
           'n_tautomeros': len(tautomeros)}

    G_list, smi_list = [], []
    directorio = f'bosque/{nombre_mol}'
    os.makedirs(directorio, exist_ok=True)

    for j, tau in enumerate(tautomeros):
        etiqueta = f'{nombre_mol}_T{j+1:02d}'
        tau_h    = Chem.AddHs(tau)
        params   = AllChem.ETKDGv3(); params.randomSeed = 42
        ok       = AllChem.EmbedMolecule(tau_h, params)
        if ok == -1:
            continue

        ff = AllChem.MMFFGetMoleculeForceField(
                 tau_h, AllChem.MMFFGetMoleculeProperties(tau_h))
        if ff is None:
            ff = AllChem.UFFGetMoleculeForceField(tau_h)
        if ff:
            ff.Minimize(maxIts=2000)

        xyz_in = f'{etiqueta}_FF.xyz'
        rdmolfiles.MolToXYZFile(tau_h,
            os.path.join(directorio, xyz_in))

        result = subprocess.run(
            ['xtb', xyz_in, '--opt', 'tight', '--ohess',
             '--chrg', '0', '--uhf', '0', '--gfn', '2',
             '--T', '298.15'],
            capture_output=True, text=True, cwd=directorio)

        m = re.search(r'TOTAL FREE ENERGY\s+([-\d.]+)\s+Eh',
                      result.stdout)
        if m:
            G_list.append(float(m.group(1)))
            smi_list.append(Chem.MolToSmiles(tau))

    if not G_list:
        resultados.append(reg); continue

    G_arr  = np.array(G_list)
    dG_kJ  = (G_arr - G_arr.min()) * 2625.5
    pesos  = np.exp(-dG_kJ / (R * T))
    pop    = pesos / pesos.sum() * 100
    orden  = np.argsort(dG_kJ)

    reg['smiles_T1']   = smi_list[orden[0]]
    reg['dG_T1_kJ']    = 0.0
    reg['dG_T2_kJ']    = dG_kJ[orden[1]] if len(orden) > 1 else None
    reg['pop_T1_pct']  = pop[orden[0]]
    reg['pop_T2_pct']  = pop[orden[1]]  if len(orden) > 1 else None
    resultados.append(reg)

pd.DataFrame(resultados).to_csv(
    'p02_bosque_resultados.csv', index=False)
print('Bosque completado.')

### Dataset pre-calculado

El bosque completo ($N = 40$ moléculas) se proporciona en
'p02\_bosque\_resultados.csv'. El estudiante añade su semilla
(2-piridona) como fila adicional antes del análisis, verificando que
sus valores de $\Delta G$ y $p_1$ son consistentes con el rango del
bosque.

## Construcción del dataset

### Integrar la semilla al dataset

In [ ]:
import pandas as pd

df = pd.read_csv('p02_bosque_resultados.csv')

semilla = {
    'id'              : '2-piridona',
    'smiles_entrada'  : 'O=c1ccccn1H',
    'familia'         : 'heterociclo_ceto_enol',
    'n_tautomeros'    : ...,   # completar
    'smiles_T1'       : ...,   # completar (SMILES del más estable)
    'dG_T1_kJ'        : 0.0,
    'dG_T2_kJ'        : ...,   # completar
    'pop_T1_pct'      : ...,   # completar
    'pop_T2_pct'      : ...,   # completar
    'dG_ref_kJ'       : -6.3,  # valor experimental de fase gas
    'referencia'      : 'beak1987',
}

df_final = pd.concat([df, pd.DataFrame([semilla])],
                     ignore_index=True)
df_final.to_csv('p02_dataset_final.csv', index=False)

## Análisis de resultados

### Estadística descriptiva

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import numpy as np

df = pd.read_csv('p02_dataset_final.csv')

print("=== Resumen del bosque ===")
print(df[['n_tautomeros','dG_T2_kJ',
          'pop_T1_pct']].describe().round(2))
print(f"\nMoléculas con T1 > 99
      f"{(df['pop_T1_pct'] > 99).sum()}")
print(f"Moléculas con T1 < 70
      f"{(df['pop_T1_pct'] < 70).sum()}")

### Gráficos principales

1. **Gráfico de barras: distribución de poblaciones del bosque.**  Pregunta: ¿qué fracción de las 40 moléculas tienen un tautómero  dominante ($p_1 > 95\  tautómera significativa ($p_1 < 80\  verbatim fig, ax = plt.subplots(figsize=(10, 4)) colores = ['#0F2747' if p > 95 else  '#2A6F97' if p > 80 else  '#AABBD0'  for p in df['pop_T1_pct']] ax.bar(range(len(df)), df['pop_T1_pct'], color=colores,  edgecolor='white', linewidth=0.5) ax.axhline(95, color='gray', linestyle='–',  linewidth=0.8, label='95 ax.axhline(80, color='gray', linestyle=':',  linewidth=0.8, label='80 ax.set_xlabel('Molécula (ordenada por población)') ax.set_ylabel('Población del tautómero más estable ( ax.legend() plt.tight_layout() plt.savefig('p02_poblaciones_barra.pdf') verbatim
2. **Dispersión: $\Delta G_{T2}$ vs $p_1$, coloreada por familia.**  Pregunta: ¿existe una familia estructural que concentre los casos  de mayor incertidumbre tautómera?  verbatim fig, ax = plt.subplots(figsize=(6, 5)) familias = df['familia'].unique() for fam in familias:  sub = df[df['familia'] == fam]  ax.scatter(sub['dG_T2_kJ'], sub['pop_T1_pct'],  label=fam, alpha=0.8, s=50,  edgecolors='k', linewidths=0.3) ax.set_xlabel(r'$\Delta G_{T_2}$ / kJ mol$^{-1}$') ax.set_ylabel('Población $p_1$ ( ax.legend(fontsize=7, loc='lower right') plt.tight_layout() plt.savefig('p02_dG_vs_pop.pdf') verbatim
3. **Validación: $\Delta G$ calculado vs experimental.**  Para las moléculas del bosque con valor de referencia disponible,  construir un gráfico de paridad.  Pregunta: ¿dónde falla GFN2-xTB respecto a los datos experimentales?  verbatim df_val = df.dropna(subset=['dG_ref_kJ']) fig, ax = plt.subplots(figsize=(5, 5)) ax.scatter(df_val['dG_ref_kJ'], df_val['dG_T2_kJ'],  edgecolors='k', linewidths=0.4, s=60) lim = max(abs(df_val[['dG_ref_kJ','dG_T2_kJ']]  .values.flatten())) + 2 ax.plot([-lim, lim], [-lim, lim], 'k–', linewidth=0.8) ax.set_xlabel(r'$\Delta G_\mathrm{exp}$ / kJ mol$^{-1}$') ax.set_ylabel(r'$\Delta G_\mathrm{calc}$ / kJ mol$^{-1}$') plt.tight_layout() plt.savefig('p02_paridad.pdf') verbatim

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
colores = ['#0F2747' if p > 95 else
           '#2A6F97' if p > 80 else
           '#AABBD0'
           for p in df['pop_T1_pct']]
ax.bar(range(len(df)), df['pop_T1_pct'], color=colores,
       edgecolor='white', linewidth=0.5)
ax.axhline(95, color='gray', linestyle='--',
           linewidth=0.8, label='95
ax.axhline(80, color='gray', linestyle=':',
           linewidth=0.8, label='80
ax.set_xlabel('Molécula (ordenada por población)')
ax.set_ylabel('Población del tautómero más estable (
ax.legend()
plt.tight_layout()
plt.savefig('p02_poblaciones_barra.pdf')

**Dispersión: $\Delta G_{T2}$ vs $p_1$, coloreada por familia.**
 Pregunta: ¿existe una familia estructural que concentre los casos
 de mayor incertidumbre tautómera?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
familias = df['familia'].unique()
for fam in familias:
    sub = df[df['familia'] == fam]
    ax.scatter(sub['dG_T2_kJ'], sub['pop_T1_pct'],
               label=fam, alpha=0.8, s=50,
               edgecolors='k', linewidths=0.3)
ax.set_xlabel(r'$\Delta G_{T_2}$ / kJ mol$^{-1}$')
ax.set_ylabel('Población $p_1$ (
ax.legend(fontsize=7, loc='lower right')
plt.tight_layout()
plt.savefig('p02_dG_vs_pop.pdf')

**Validación: $\Delta G$ calculado vs experimental.**
 Para las moléculas del bosque con valor de referencia disponible,
 construir un gráfico de paridad.
 Pregunta: ¿dónde falla GFN2-xTB respecto a los datos experimentales?

In [ ]:
df_val = df.dropna(subset=['dG_ref_kJ'])
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(df_val['dG_ref_kJ'], df_val['dG_T2_kJ'],
           edgecolors='k', linewidths=0.4, s=60)
lim = max(abs(df_val[['dG_ref_kJ','dG_T2_kJ']]
              .values.flatten())) + 2
ax.plot([-lim, lim], [-lim, lim], 'k--', linewidth=0.8)
ax.set_xlabel(r'$\Delta G_\mathrm{exp}$ / kJ mol$^{-1}$')
ax.set_ylabel(r'$\Delta G_\mathrm{calc}$ / kJ mol$^{-1}$')
plt.tight_layout()
plt.savefig('p02_paridad.pdf')

enumerate

## Interpretación química

Para la semilla, el resultado de GFN2-xTB debería confirmar que la
2-piridona (forma lactámica) es más estable que la 2-hidroxipiridina
en fase gas, con $\Delta G \approx -5$~a~$-8$~kJ/mol. Esto sitúa la
población de la forma ceto en torno al 88–97\
con el valor experimental de Beak cite'beak1987'. La estabilización de
la forma ceto se debe a la conjunción de dos efectos: la aromaticidad
del anillo lactámico (el par del N estabiliza el sistema $\pi$
delocalizado) y la polaridad del enlace C=O (que permite interacciones
electrostáticas favorables incluso en vacío).

En el bosque, el análisis por familia debería revelar que los
*heterociclos simples con un solo grupo tautomerizable* son los
más predecibles ($p_1 > 95\
que los *fármacos con múltiples centros tautomerizables*
muestran la mayor dispersión de poblaciones. Las bases nucleicas
presentan un caso intermedio: la forma canónica (keto-amino) domina
ampliamente para adenina y guanina, pero los tautómeros imino pueden
alcanzar poblaciones del 1–5\
si se desproducen durante la replicación.

La comparación con datos experimentales en el gráfico de paridad
revelará probablemente que GFN2-xTB subestima $|\Delta G|$ para
sistemas muy polarizados (donde el efecto del solvente es crítico)
y lo sobreestima para compuestos con extensa conjugación $\pi$.
Ambas limitaciones son esperables: GFN2-xTB es un método de vacío y
su descripción de la deslocalización $\pi$ tiene la precisión de
un semiempírico, no de un método DFT de alto nivel. La corrección
de ambas limitaciones se introduce en la Práctica~28 (PCM/SMD) y en
el Bloque~6 (post-HF), respectivamente.

```{admonition} 📝 Nota metodológica
:class: note

**Conexión con prácticas posteriores:** El dataset de tautómeros
dominantes generado aquí es el input correcto para los cálculos de
estructura electrónica del Bloque~2. Usar la forma tautómera incorrecta
en la Práctica~4 produciría errores sistemáticos en todas las
propiedades calculadas. El efecto del solvente sobre los equilibrios
tautómeros se cuantificará en la Práctica~30.
```

## Entregables

### Datos

- ****D1.**** '2-piridona\_tautomeros\_resultados.csv'  — tabla con $G$, $\Delta G$ y poblaciones de todos los  tautómeros de la semilla.
- ****D2.**** 'p02\_dataset\_final.csv' — bosque  completo con la fila de la semilla añadida.
- ****D3.**** '2-piridona\_T01\_xtb.out' y  'T02\_xtb.out' — logs de xTB del tautómero dominante  y del segundo más estable.

### Figuras

- ****F1.**** 'p02\_poblaciones\_barra.pdf' —  distribución de $p_1$ para el bosque completo.
- ****F2.**** 'p02\_dG\_vs\_pop.pdf' —  dispersión $\Delta G_{T2}$ vs $p_1$, coloreada por familia.
- ****F3.**** 'p02\_paridad.pdf' —  gráfico de paridad calculado vs experimental.
- ****F4.**** (Opcional) Imagen de la rejilla 2D de los  tautómeros de la semilla con sus poblaciones anotadas.

### Texto

- ****T1.**** Reporte ($\leq 2$ páginas): tabla de validación  semilla vs referencia, F1 y F2 comentadas, interpretación  del equilibrio tautómero de la 2-piridona.
- ****T2.**** Respuestas a las preguntas de discusión  ($\leq 150$ palabras por pregunta).

## Preguntas de discusión

1. **(Comprensión)** ¿Cuántos tautómeros encontró  'TautomerEnumerator' para la 2-piridona? ¿Son todos  químicamente razonables? Si alguno no lo es, explica por  qué el algoritmo lo genera a pesar de eso.
2. **(Comprensión)** Examina el log de xTB del tautómero  dominante e identifica la línea que reporta la energía libre  de Gibbs $G(298\,\mathrm{K})$. ¿Qué contribuciones  energéticas suma xTB para obtener ese valor?  ¿Qué diferencia hay entre $E_\mathrm{total}$ y  $G(298\,\mathrm{K})$?
3. **(Análisis)** Usa la ecuación de Boltzmann para calcular  manualmente qué diferencia de $\Delta G$ sería necesaria para  que el tautómero minoritario tuviera una población de  exactamente 1\  el tautómero minoritario de la 2-piridona con una  técnica espectroscópica estándar?
4. **(Análisis)** Del bosque, identifica la molécula con la  distribución de Boltzmann más "plana" (menor $\Delta G_{T2}$,  mayor incertidumbre tautómera). ¿Qué consecuencia tendría  para un cálculo de docking que se ejecutara sobre el  tautómero incorrecto de esa molécula?
5. **(Metodológico)** El protocolo usa energías en vacío.  ¿En qué dirección esperarías que se desplazara el equilibrio  2-piridona $\rightleftharpoons$ 2-hidroxipiridina al pasar  de vacío a agua? ¿A ciclohexano? Justifica usando argumentos  de polaridad y de puentes de hidrógeno.
6. **(Metodológico)** RDKit enumera tautómeros basándose  en reglas SMARTS predefinidas. ¿Qué tipo de tautomería  podría *no* detectar el algoritmo? Propón un ejemplo  de una transformación tautómera que no esté cubierta por las  reglas estándar de Sitzmann.

## Extensiones del ejercicio

- **Temperatura y equilibrio:** recalcular las poblaciones  de Boltzmann de la semilla a 200, 298, 400 y 600~K usando los  mismos valores de $G$ (aproximación de geometría congelada).  Graficar $p_1(T)$ y determinar la temperatura a la que ambas  formas estarían igualmente pobladas.
- **Bases nucleicas completas:** ampliar el bosque con los  cuatro tautómeros relevantes de la guanina y construir el  diagrama de energías relativas. Comparar con los valores  reportados por Mata *et al.* (2010) a nivel  CCSD(T)/CBS.
- **Efecto del solvente:** añadir el modelo GBSA-H$_2$O  de xTB ('–alpb water') para recalcular $\Delta G$ en  agua implícita. Comparar con los valores en vacío y con el dato  experimental en agua de Katritzky (1991). Esto anticipa la  Práctica~28.
- **Pipeline de normalización molecular:** integrar el  tautómero dominante calculado en este bosque como paso de  preprocesamiento en el pipeline de la Práctica~47 (exploración  del espacio químico). ¿Cuántas moléculas del bosque de la  Práctica~47 cambiarían su SMILES canónico si se usara el  tautómero más estable en lugar del SMILES de entrada original?

## 📚 Referencias

- **[abrahamsson2024]** Abrahamsson, D. *Modeling the relative response factor of
small molecules in positive electrospray ionization mass spectrometry
using computationally generated tautomers and protomers*.
*RSC Adv.* **14**, 31350–31361, 2024.
DOI: [DOI:10.1039/d4ra06695b](https://doi.org/10.1039/d4ra06695b).
- **[beak1987]** Beak, P.; Fry, F.~S.; Lee, J.; Steele, F.
*Protomeric equilibria of 2- and 4-hydroxypyridines and
of 2- and 4-pyridinethiols in the gas phase and in solution*.
*J. Am. Chem. Soc.* **98**(1), 171–179, 1976.
DOI: [DOI:10.1021/ja00417a028](https://doi.org/10.1021/ja00417a028).
- **[kaur2021]** Kaur, R.; Choudhury, A.~R. *Tautomerism in drug molecules:
the forgotten facet of drug design*. *J. Med. Chem.*
**64**(8), 4311–4325, 2021.
DOI: [DOI:10.1021/acs.jmedchem.0c01989](https://doi.org/10.1021/acs.jmedchem.0c01989).
- **[rdkit_p02]** Landrum, G. *RDKit: Open-Source Cheminformatics*.
Disponible en: [https://www.rdkit.org](https://www.rdkit.org). Acceso: 2026.
- **[grimme_xtb_p02]** Bannwarth, C.; Ehlert, S.; Grimme, S. *GFN2-xTB — An
Accurate and Broadly Parametrized Self-Consistent Tight-Binding
Quantum Chemical Method with Multipole Electrostatics and
Density-Dependent Dispersion Contributions*.
*J. Chem. Theory Comput.* **15**(3), 1652–1671, 2019.
DOI: [DOI:10.1021/acs.jctc.8b01176](https://doi.org/10.1021/acs.jctc.8b01176).
- **[lewars_p02]** Lewars, E.~G. *Computational Chemistry: Introduction to
the Theory and Applications of Molecular and Quantum Mechanics*.
Springer, 2024. DOI: [DOI:10.1007/978-3-031-51443-2](https://doi.org/10.1007/978-3-031-51443-2).